In [1]:
from nnsight import LanguageModel
import torch


In [2]:
import einops

In [3]:
from nnsight import CONFIG
CONFIG.API.APIKEY = "c082b87e-d32b-426f-961f-bd636a711885"

In [4]:
# Load MATH-500 dataset using direct HTTP
import requests
import pandas as pd

# Get all 500 rows by making multiple API calls (limit is 100 per call)
all_rows = []
for offset in range(0, 500, 100):
    api_url = f"https://datasets-server.huggingface.co/rows?dataset=HuggingFaceH4/MATH-500&config=default&split=test&offset={offset}&length=100"
    response = requests.get(api_url)
    data = response.json()
    all_rows.extend([row['row'] for row in data['rows']])

# Convert to DataFrame
math_df = pd.DataFrame(all_rows)

print(f"Loaded {len(math_df)} problems")
print(f"Columns: {list(math_df.columns)}")
print(f"\nSample problem:")
print(f"Problem: {math_df.iloc[0]['problem']}")
print(f"Solution: {math_df.iloc[0]['solution'][:200]}...")


Loaded 500 problems
Columns: ['problem', 'solution', 'answer', 'subject', 'level', 'unique_id']

Sample problem:
Problem: Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\theta),$ where $r > 0$ and $0 \le \theta < 2 \pi.$
Solution: We have that $r = \sqrt{0^2 + 3^2} = 3.$  Also, if we draw the line connecting the origin and $(0,3),$ this line makes an angle of $\frac{\pi}{2}$ with the positive $x$-axis.

[asy]
unitsize(0.8 cm);
...


In [5]:
math_df.head()

,problem,solution,answer,subject,level,unique_id
0,"Convert the point $(0,3)$ in rectangular coord...",We have that $r = \sqrt{0^2 + 3^2} = 3.$ Also...,"\left( 3, \frac{\pi}{2} \right)",Precalculus,2,test/precalculus/807.json
1,Define\n\[p = \sum_{k = 1}^\infty \frac{1}{k^2...,We count the number of times $\frac{1}{n^3}$ a...,p - q,Intermediate Algebra,5,test/intermediate_algebra/1994.json
2,"If $f(x) = \frac{3x-2}{x-2}$, what is the valu...",$f(-2)+f(-1)+f(0)=\frac{3(-2)-2}{-2-2}+\frac{3...,\frac{14}{3},Algebra,3,test/algebra/2584.json
3,How many positive whole-number divisors does 1...,First prime factorize $196=2^2\cdot7^2$. The ...,9,Number Theory,3,test/number_theory/572.json
4,The results of a cross-country team's training...,Evelyn covered more distance in less time than...,\text{Evelyn},Algebra,2,test/algebra/1349.json


In [6]:
# Plan
# 1) Load first MATH-500 problem from existing df or fetch if missing
# 2) Initialize DeepSeek-R1-Distill-Qwen-1.5B locally
# 3) Build prompt with explicit thinking and boxed answer instructions
# 4) Generate 100 samples locally with moderate temperature/top_p
# 5) During generation, collect per-step last-layer hidden states and tokens
# 6) Decode, parse second <think> block, response after </think>, and final \boxed{...}
# 7) Compute combined hidden tensor batch size and assemble DataFrame

import re
import math
import json
import torch
import pandas as pd
from nnsight import LanguageModel

# Ensure math_df exists; if not, fetch quickly
try:
    _ = math_df
except NameError:
    import requests
    all_rows = []
    for offset in range(0, 100, 100):
        api_url = f"https://datasets-server.huggingface.co/rows?dataset=HuggingFaceH4/MATH-500&config=default&split=test&offset={offset}&length=100"
        data = requests.get(api_url).json()
        all_rows.extend([row['row'] for row in data['rows']])
    math_df = pd.DataFrame(all_rows)

first_problem = math_df.iloc[0]['problem']


In [7]:
model=LanguageModel("deepseek-ai/DeepSeek-R1-Distill-Llama-8B", device_map="auto")
print(model.model)

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

LlamaModel(
  (embed_tokens): Embedding(128256, 4096)
  (layers): ModuleList(
    (0-31): 32 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
        (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
        (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
    )
  )
  (norm): LlamaRMSNorm((4096,), eps=1e-05)
  (rotary_emb): LlamaRotaryEmbedding()
)


In [8]:
import os

In [9]:

def run_and_save_for_problem(problem_text, problem_id, num_samples=20, max_new_tokens=512, save_root="/workspace/uncertainty/rollouts"):
# Build prompt with thinking and boxed answer instructions
    os.makedirs(save_root, exist_ok=True)
    system_instructions = f"Solve the following math problem. Enclose your thoughts within <think>\n\n</think> tags. Your final answer should be denoted as **Answer:** and must be enclosed in \\boxed{{}}"
    chat = [
        {"role": "user", "content": f"{system_instructions}\n\nProblem: {problem_text}"},
    ]
    prompt = model.tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True)

    num_samples = 35
    max_new_tokens = 2048


    # Capture hidden states for every new token across the whole batch of 100 prompts


    # Prepare batch prompts
    batch_prompts = [prompt] * num_samples
    promptlen = len(prompt)
    with model.generate(max_new_tokens=max_new_tokens, temperature=0.6, top_p=0.95, remote=True, do_sample=True) as tracer:
            with tracer.invoke(batch_prompts):
                hidden_collector = list().save()
                with tracer.all():
                # last layer hidden states for all sequences at current step
                    hidden_collector.append(model.model.layers[-1].output)
                # save generated token ids per sequence
            with tracer.invoke():
                token_batch = model.generator.output.save()
    hidden_states=torch.cat(hidden_collector, dim=1)
    keep_rows = (token_batch[:, promptlen:] == 128014).any(dim=1)
    token_batch = token_batch[keep_rows]
    b_idx, t_rel = (token_batch[:, promptlen:] == 128014).nonzero(as_tuple=True)
    t_abs = t_rel + promptlen
    hidden_states = hidden_states[b_idx, t_abs]
    sample = {
        "problem_id": problem_id,
        "hidden_states": hidden_states.cpu(),
        "token_batch": token_batch.cpu(),
        "system_instructions": system_instructions,
        "prompt": prompt,
    }            
    torch.save(sample, os.path.join(save_root, f"{problem_id}.pt"))        


In [15]:
len(math_df)


500

In [11]:
problem_text= math_df['problem'][1]
problem_id= math_df['unique_id'][1].replace('/', '')


In [12]:
math_df

,problem,solution,answer,subject,level,unique_id
0,"Convert the point $(0,3)$ in rectangular coord...",We have that $r = \sqrt{0^2 + 3^2} = 3.$ Also...,"\left( 3, \frac{\pi}{2} \right)",Precalculus,2,test/precalculus/807.json
1,Define\n\[p = \sum_{k = 1}^\infty \frac{1}{k^2...,We count the number of times $\frac{1}{n^3}$ a...,p - q,Intermediate Algebra,5,test/intermediate_algebra/1994.json
2,"If $f(x) = \frac{3x-2}{x-2}$, what is the valu...",$f(-2)+f(-1)+f(0)=\frac{3(-2)-2}{-2-2}+\frac{3...,\frac{14}{3},Algebra,3,test/algebra/2584.json
3,How many positive whole-number divisors does 1...,First prime factorize $196=2^2\cdot7^2$. The ...,9,Number Theory,3,test/number_theory/572.json
4,The results of a cross-country team's training...,Evelyn covered more distance in less time than...,\text{Evelyn},Algebra,2,test/algebra/1349.json
...,...,...,...,...,...,...
495,What is the domain of the function $f(x) = \fr...,The inner logarithm is only defined if $x - 2 ...,"(2,12) \cup (12,102)",Intermediate Algebra,4,test/intermediate_algebra/1981.json
496,Let $z = 1+i$ and $w = \dfrac{3z+1}{5z+7}$. Fi...,"Plugging in, we have $w = \dfrac{3(1+i)+1}{5(1...",\frac{5}{13},Intermediate Algebra,3,test/intermediate_algebra/1232.json
497,An equiangular octagon has four sides of lengt...,The octagon can be partitioned into five squar...,\frac{7}{2},Geometry,5,test/geometry/561.json
498,A sequence $(a_n)$ is defined as follows:\n\[a...,"First, if $a_3 = a_1,$ then\n\[a_1 = a_3 = a_5...",-1,Intermediate Algebra,5,test/intermediate_algebra/1508.json


In [14]:
for i in range(75,len(math_df)):
    try:
        problem_text= math_df['problem'][i]
        problem_id= math_df['unique_id'][i].replace('/', '')
        run_and_save_for_problem(problem_text, problem_id)
    except Exception as e:
        print(f"Error on problem {i}: {e}")
        continue

[2025-08-29 14:19:43] [cbc45acf-4df5-44aa-bbde-efe42eaa9499] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-08-29 14:19:43] [cbc45acf-4df5-44aa-bbde-efe42eaa9499] QUEUED     : Your job has been recieved by the coordinator and is waiting to be queued.
[2025-08-29 14:19:43] [cbc45acf-4df5-44aa-bbde-efe42eaa9499] QUEUED     : Task cbc45acf-4df5-44aa-bbde-efe42eaa9499 has been added to the queue. Currently at position 1
[2025-08-29 14:19:46] [cbc45acf-4df5-44aa-bbde-efe42eaa9499] QUEUED     : `nnsight.modeling.language.LanguageModel:{"repo_id": "deepseek-ai/DeepSeek-R1-Distill-Llama-8B", "revision": "main"}` is being deployed... stand by.
[2025-08-29 14:19:47] [cbc45acf-4df5-44aa-bbde-efe42eaa9499] QUEUED     : `nnsight.modeling.language.LanguageModel:{"repo_id": "deepseek-ai/DeepSeek-R1-Distill-Llama-8B", "revision": "main"}` is being deployed... stand by.
[2025-08-29 14:19:48] [cbc45acf-4df5-44aa-bbde-efe42eaa9499] QUEUED     : `nnsight.modeling.language.Langu

[2025-08-29 14:23:30] [c860fe18-f025-4502-ac7e-c3a218af1ca1] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-08-29 14:23:30] [c860fe18-f025-4502-ac7e-c3a218af1ca1] QUEUED     : Your job has been recieved by the coordinator and is waiting to be queued.
[2025-08-29 14:23:30] [c860fe18-f025-4502-ac7e-c3a218af1ca1] QUEUED     : Task c860fe18-f025-4502-ac7e-c3a218af1ca1 has been added to the queue. Currently at position 1
[2025-08-29 14:23:30] [c860fe18-f025-4502-ac7e-c3a218af1ca1] QUEUED     : Dispatching task...
[2025-08-29 14:23:30] [c860fe18-f025-4502-ac7e-c3a218af1ca1] RUNNING    : Your job has started running.
[2025-08-29 14:26:01] [c860fe18-f025-4502-ac7e-c3a218af1ca1] COMPLETED  : Your job has been completed.


[2025-08-29 14:26:32] [449ec6bf-6396-46c8-b7bd-f4dd9ccf6810] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-08-29 14:26:32] [449ec6bf-6396-46c8-b7bd-f4dd9ccf6810] QUEUED     : Your job has been recieved by the coordinator and is waiting to be queued.
[2025-08-29 14:26:32] [449ec6bf-6396-46c8-b7bd-f4dd9ccf6810] QUEUED     : Task 449ec6bf-6396-46c8-b7bd-f4dd9ccf6810 has been added to the queue. Currently at position 1
[2025-08-29 14:26:32] [449ec6bf-6396-46c8-b7bd-f4dd9ccf6810] QUEUED     : Dispatching task...
[2025-08-29 14:26:34] [449ec6bf-6396-46c8-b7bd-f4dd9ccf6810] RUNNING    : Your job has started running.
[2025-08-29 14:27:00] [449ec6bf-6396-46c8-b7bd-f4dd9ccf6810] COMPLETED  : Your job has been completed.


[2025-08-29 14:27:08] [ac0de300-543f-43fa-803a-149f14c499b0] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-08-29 14:27:08] [ac0de300-543f-43fa-803a-149f14c499b0] QUEUED     : Your job has been recieved by the coordinator and is waiting to be queued.
[2025-08-29 14:27:08] [ac0de300-543f-43fa-803a-149f14c499b0] QUEUED     : Task ac0de300-543f-43fa-803a-149f14c499b0 has been added to the queue. Currently at position 1
[2025-08-29 14:27:08] [ac0de300-543f-43fa-803a-149f14c499b0] QUEUED     : Dispatching task...
[2025-08-29 14:27:08] [ac0de300-543f-43fa-803a-149f14c499b0] RUNNING    : Your job has started running.
[2025-08-29 14:29:41] [ac0de300-543f-43fa-803a-149f14c499b0] COMPLETED  : Your job has been completed.


[2025-08-29 14:30:07] [d21bfc8e-856b-4bf8-a3a9-09e40c90a696] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-08-29 14:30:07] [d21bfc8e-856b-4bf8-a3a9-09e40c90a696] QUEUED     : Your job has been recieved by the coordinator and is waiting to be queued.
[2025-08-29 14:30:07] [d21bfc8e-856b-4bf8-a3a9-09e40c90a696] QUEUED     : Task d21bfc8e-856b-4bf8-a3a9-09e40c90a696 has been added to the queue. Currently at position 1
[2025-08-29 14:30:07] [d21bfc8e-856b-4bf8-a3a9-09e40c90a696] QUEUED     : Dispatching task...
[2025-08-29 14:30:07] [d21bfc8e-856b-4bf8-a3a9-09e40c90a696] RUNNING    : Your job has started running.
[2025-08-29 14:30:30] [d21bfc8e-856b-4bf8-a3a9-09e40c90a696] COMPLETED  : Your job has been completed.


...

Error on problem 236: Traceback (most recent call last):
  File "/u/svcndifuser/ndif-deployment/repos/prod/ndif/src/services/ray/src/ray/deployments/modeling/base.py", line 213, in __call__
    result = await job_task
  File "/u/svcndifuser/miniconda3/lib/python3.10/asyncio/threads.py", line 25, in to_thread
    return await loop.run_in_executor(None, func_call)
  File "/u/svcndifuser/miniconda3/lib/python3.10/concurrent/futures/thread.py", line 58, in run
    result = self.fn(*self.args, **self.kwargs)
  File "/u/svcndifuser/miniconda3/lib/python3.10/site-packages/ray/util/tracing/tracing_helper.py", line 463, in _resume_span
    return method(self, *_args, **_kwargs)
  File "/u/svcndifuser/ndif-deployment/repos/prod/ndif/src/services/ray/src/ray/deployments/modeling/base.py", line 283, in execute
    result = RemoteExecutionBackend(request.interventions, self.execution_protector)(request.tracer)
  File "/u/svcndifuser/ndif-deployment/repos/prod/ndif/src/services/ray/src/ray/nn/backen

[2025-08-29 19:17:47] [dcab0d97-4a77-4852-bd3e-da056cb7fa40] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-08-29 19:17:47] [dcab0d97-4a77-4852-bd3e-da056cb7fa40] QUEUED     : Your job has been recieved by the coordinator and is waiting to be queued.
[2025-08-29 19:17:47] [dcab0d97-4a77-4852-bd3e-da056cb7fa40] QUEUED     : Task dcab0d97-4a77-4852-bd3e-da056cb7fa40 has been added to the queue. Currently at position 1
[2025-08-29 19:17:47] [dcab0d97-4a77-4852-bd3e-da056cb7fa40] QUEUED     : Dispatching task...
[2025-08-29 19:17:48] [dcab0d97-4a77-4852-bd3e-da056cb7fa40] RUNNING    : Your job has started running.


Error on problem 237: Traceback (most recent call last):
  File "/u/svcndifuser/ndif-deployment/repos/prod/ndif/src/services/ray/src/ray/deployments/modeling/base.py", line 213, in __call__
    result = await job_task
  File "/u/svcndifuser/miniconda3/lib/python3.10/asyncio/threads.py", line 25, in to_thread
    return await loop.run_in_executor(None, func_call)
  File "/u/svcndifuser/miniconda3/lib/python3.10/concurrent/futures/thread.py", line 58, in run
    result = self.fn(*self.args, **self.kwargs)
  File "/u/svcndifuser/miniconda3/lib/python3.10/site-packages/ray/util/tracing/tracing_helper.py", line 463, in _resume_span
    return method(self, *_args, **_kwargs)
  File "/u/svcndifuser/ndif-deployment/repos/prod/ndif/src/services/ray/src/ray/deployments/modeling/base.py", line 283, in execute
    result = RemoteExecutionBackend(request.interventions, self.execution_protector)(request.tracer)
  File "/u/svcndifuser/ndif-deployment/repos/prod/ndif/src/services/ray/src/ray/nn/backen

[2025-08-29 19:18:06] [002fd471-dbed-4216-8426-93a4af327e83] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-08-29 19:18:06] [002fd471-dbed-4216-8426-93a4af327e83] QUEUED     : Your job has been recieved by the coordinator and is waiting to be queued.
[2025-08-29 19:18:06] [002fd471-dbed-4216-8426-93a4af327e83] QUEUED     : Task 002fd471-dbed-4216-8426-93a4af327e83 has been added to the queue. Currently at position 1
[2025-08-29 19:18:06] [002fd471-dbed-4216-8426-93a4af327e83] QUEUED     : Dispatching task...
[2025-08-29 19:18:06] [002fd471-dbed-4216-8426-93a4af327e83] RUNNING    : Your job has started running.


KeyboardInterrupt: 

In [ ]:
print(token_batch.shape)
print(token_batch[0][-1])
print(hidden_collector[-1].shape)
torch.cat(hidden_collector, dim=1).shape


In [ ]:
model.tokenizer.decode(token_batch[0])

In [ ]:
hidden_states=torch.cat(hidden_collector, dim=1)
keep_rows = (token_batch[:, promptlen:] == 128014).any(dim=1)
token_batch = token_batch[keep_rows]
b_idx, t_rel = (token_batch[:, promptlen:] == 128014).nonzero(as_tuple=True)
t_abs = t_rel + promptlen
hidden_states = hidden_states[b_idx, t_abs]

In [ ]:
hidden_states.shape

In [ ]:
import glob

In [ ]:
paths = sorted(glob.glob("/workspace/uncertainty/rollouts/*.pt"))
samples = [torch.load(p, map_location="cpu") for p in paths]



In [ ]:
generated_token_ids = []
think_blocks = []
responses = []
final_answers = []

# Post-process hidden states

# Helpers
_defective_close_variants = ["<\\think>", "< /think>", "</ think>"]

def _normalize_think_tags(text: str) -> str:
    out = text
    for bad in _defective_close_variants:
        out = out.replace(bad, "</think>")
    return out

def extract_second_think_and_response(text: str):
    norm = _normalize_think_tags(text)
    matches = list(re.finditer(r"<think>([\s\S]*?)</think>", norm))
    if len(matches) >= 2:
        second = matches[1]
        return matches[1].group(1).strip(), norm[second.end():].strip()
    if len(matches) == 1:
        first = matches[0]
        return first.group(1).strip(), norm[first.end():].strip()
    return "", norm.strip()

def extract_boxed_balanced(text: str):
    m = re.search(r"\\boxed\s*\{", text)
    if not m:
        return ""
    i = m.end()
    depth = 1
    while i < len(text) and depth > 0:
        ch = text[i]
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
        i += 1
    if depth == 0:
        return text[m.end(): i - 1].strip()
    return ""

# Decode and parse
token_ids_per_seq = token_batch
for i in range(len(token_ids_per_seq)):
    full_text = model.tokenizer.decode(token_ids_per_seq[i], skip_special_tokens=False)
    gen_only = full_text[len(prompt):]

    think_text, response_after = extract_second_think_and_response(gen_only)
    final = extract_boxed_balanced(response_after) or extract_boxed_balanced(gen_only)

    generated_token_ids.append(token_ids_per_seq[i])
    think_blocks.append(think_text)
    responses.append(response_after)
    final_answers.append(final)

results_df = pd.DataFrame({
    "think": think_blocks,
    "response": responses,
    "final": final_answers,
    "num_tokens": [len(x) for x in generated_token_ids],
})

print("Rows:", len(results_df))
results_df.head()


In [ ]:
results_df['response'][0]

In [ ]:
results_df['think'][0]